In [60]:
import pandas as pd
df = pd.read_csv("final_combined_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   text           4000 non-null   object 
 1   sentiment      4000 non-null   object 
 2   OrderID        4000 non-null   object 
 3   CustomerName   4000 non-null   object 
 4   CustomerEmail  4000 non-null   object 
 5   City           4000 non-null   object 
 6   OrderDate      4000 non-null   object 
 7   Product        4000 non-null   object 
 8   Quantity       4000 non-null   int64  
 9   Discount(%)    4000 non-null   int64  
 10  TotalPrice($)  4000 non-null   float64
 11  PaymentMethod  4000 non-null   object 
 12  Shipped        4000 non-null   object 
dtypes: float64(1), int64(2), object(10)
memory usage: 406.4+ KB


In [61]:
df.head()

,text,sentiment,OrderID,CustomerName,CustomerEmail,City,OrderDate,Product,Quantity,Discount(%),TotalPrice($),PaymentMethod,Shipped
0,I would definitely recommend this to others. I...,Positive,a14da64c-3d17-4df5-9e2d-9e11228930e2,Joseph Nelson,hardydave@example.com,New Angelafurt,2025-02-19,Desk Chair,2,0,400.0,PayPal,Yes
1,I would definitely recommend this to others. I...,Positive,dc17b9cc-2b7d-4dde-bcc7-40b71ded61be,Frank Mitchell,qsmith@example.net,Port Michelle,2024-11-20,Mouse,4,0,200.0,Apple Pay,Yes
2,This was one of the worst experiences I have h...,Negative,00772932-07b1-4dcd-b95a-e6b258f12ae7,Maria Reynolds,ericmejia@example.com,New Andrew,2025-04-04,Keyboard,3,10,270.0,PayPal,Yes
3,I am completely satisfied with the outcome. I ...,Positive,5bb8b9c8-8932-49f3-9ee8-9a4c681134e2,Christine Parker,melissa12@example.net,Margaretton,2024-07-07,External Hard Drive,3,0,270.0,Debit Card,Yes
4,This was a complete waste of time and money.,Negative,7f671eec-3da7-4577-aee1-e86b9ea556aa,Briana Murray,steve07@example.com,North Scott,2024-08-24,External Hard Drive,3,0,270.0,Credit Card,Yes


In [62]:
#Drop Column
df.drop(['OrderID','CustomerEmail'], axis=1, inplace=True)

In [63]:
#Null values
df.isnull().sum()

text             0
sentiment        0
CustomerName     0
City             0
OrderDate        0
Product          0
Quantity         0
Discount(%)      0
TotalPrice($)    0
PaymentMethod    0
Shipped          0
dtype: int64

In [64]:
#Duplicate values
df.duplicated().sum()

0

In [65]:
# #Fill null values
# df["Discount(%)"] = df["Discount(%)"].isnull().median(df["Discount(%)"])
# df.fillna(df.median(numeric_only=True), inplace=True)
# df['Product'] = df['Product'].fillna("iPhone")

In [66]:
#Encoding

#Mapping
df['sentiment_encoded'] = df['sentiment'].map({
    'Negative': 1,
    'Positive': 0
})

df['Shipped_encoded'] = df['Shipped'].map({
    'Yes': 1,
    'No': 0
})


In [67]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['City_encoded'] = le.fit_transform(df['City'])

In [68]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

# 1. Create encoder
ohe = OneHotEncoder(drop='first', sparse_output=False)

# 2. Fit & transform Product column
product_encoded = ohe.fit_transform(df[['Product']])

# 3. Convert to DataFrame
product_df = pd.DataFrame(
    product_encoded,
    columns=ohe.get_feature_names_out(['Product']),
    index=df.index
)

# 4. Add encoded columns to original dataset
df = pd.concat([df, product_df], axis=1)


In [69]:
df = pd.get_dummies(
    df,
    columns=['PaymentMethod'],
    drop_first=True,
    dtype=int
)


In [70]:
df['OrderDate'] = pd.to_datetime(df['OrderDate'])

df['year'] = df['OrderDate'].dt.year
df['month'] = df['OrderDate'].dt.month
df['day'] = df['OrderDate'].dt.day
df['day_of_week'] = df['OrderDate'].dt.dayofweek


In [71]:
import pandas as pd
import string
import nltk
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_text(text):
    if pd.isna(text):
        return text
    
    # lowercase
    text = text.lower()
    
    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # remove stopwords
    text = ' '.join(word for word in text.split() if word not in stop_words)
    
    return text

df['text'] = df['text'].apply(clean_text)


In [72]:
df[['text']].head()


,text
0,would definitely recommend others felt valued ...
1,would definitely recommend others felt valued ...
2,one worst experiences issue still resolved
3,completely satisfied outcome would happily use
4,complete waste time money


In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=100)

# Fit and transform the cleaned text
tfidf_features = tfidf.fit_transform(df['text'])

# Convert to DataFrame
tfidf_df = pd.DataFrame(
    tfidf_features.toarray(),
    columns=tfidf.get_feature_names_out(),
    index=df.index
)

# Add TF-IDF features to original dataframe
df = pd.concat([df, tfidf_df], axis=1)

# Optional: drop original text column
df.drop(columns=['text'], inplace=True)


In [74]:
df.head()


,sentiment,CustomerName,City,OrderDate,Product,Quantity,Discount(%),TotalPrice($),Shipped,sentiment_encoded,...,use,using,valued,way,well,whole,without,worked,worst,would
0,Positive,Joseph Nelson,New Angelafurt,2025-02-19,Desk Chair,2,0,400.0,Yes,0,...,0.000000,0.0,0.357546,0.0,0.0,0.0,0.0,0.0,0.00000,0.305247
1,Positive,Frank Mitchell,Port Michelle,2024-11-20,Mouse,4,0,200.0,Yes,0,...,0.000000,0.0,0.357546,0.0,0.0,0.0,0.0,0.0,0.00000,0.305247
2,Negative,Maria Reynolds,New Andrew,2025-04-04,Keyboard,3,10,270.0,Yes,1,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.48338,0.000000
3,Positive,Christine Parker,Margaretton,2024-07-07,External Hard Drive,3,0,270.0,Yes,0,...,0.389758,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.00000,0.335501
4,Negative,Briana Murray,North Scott,2024-08-24,External Hard Drive,3,0,270.0,Yes,1,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.00000,0.000000


In [75]:
df.dtypes

sentiment               object
CustomerName            object
City                    object
OrderDate       datetime64[ns]
Product                 object
                     ...      
whole                  float64
without                float64
worked                 float64
worst                  float64
would                  float64
Length: 128, dtype: object

In [76]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()
print(numeric_cols)


['Quantity', 'Discount(%)', 'TotalPrice($)', 'sentiment_encoded', 'Shipped_encoded', 'City_encoded', 'Product_External Hard Drive', 'Product_Headphones', 'Product_Keyboard', 'Product_Laptop', 'Product_Monitor', 'Product_Mouse', 'Product_Smartphone', 'Product_Webcam', 'PaymentMethod_Credit Card', 'PaymentMethod_Debit Card', 'PaymentMethod_Google Pay', 'PaymentMethod_PayPal', 'year', 'month', 'day', 'day_of_week', 'best', 'better', 'caused', 'choosing', 'chose', 'completely', 'customer', 'definitely', 'delays', 'disappointed', 'disappointing', 'easy', 'enjoyable', 'everything', 'exceeded', 'excellent', 'expectations', 'expected', 'experience', 'experiences', 'extremely', 'faced', 'far', 'feeling', 'felt', 'finish', 'follow', 'frustrated', 'glad', 'great', 'handled', 'happily', 'happy', 'helpful', 'ignore', 'impressive', 'inconvenience', 'issue', 'issues', 'left', 'lot', 'made', 'many', 'meet', 'met', 'multiple', 'none', 'nothing', 'one', 'others', 'outcome', 'outstanding', 'overall', 'pe

In [77]:
X = df[['Quantity', 'Discount(%)', 'TotalPrice($)', 'sentiment_encoded', 'City_encoded', 'PaymentMethod_Credit Card', 'PaymentMethod_Debit Card', 'PaymentMethod_Google Pay', 'PaymentMethod_PayPal', 'Product_External Hard Drive', 'Product_Headphones', 'Product_Keyboard', 'Product_Laptop', 'Product_Monitor', 'Product_Mouse', 'Product_Smartphone', 'Product_Webcam', 'year', 'month', 'day', 'day_of_week', 'best', 'better', 'caused', 'choosing', 'chose', 'completely', 'customer', 'definitely', 'delays', 'disappointed', 'disappointing', 'easy', 'enjoyable', 'everything', 'exceeded', 'excellent', 'expectations', 'expected', 'experience', 'experiences', 'extremely', 'faced', 'far', 'feeling', 'felt', 'finish', 'follow', 'frustrated', 'glad', 'great', 'handled', 'happily', 'happy', 'helpful', 'ignore', 'impressive', 'inconvenience', 'issue', 'issues', 'left', 'lot', 'made', 'many', 'meet', 'met', 'multiple', 'none', 'nothing', 'one', 'others', 'outcome', 'outstanding', 'overall', 'perfectly', 'pleasant', 'pleased', 'poor', 'poorly', 'problems', 'process', 'professionally', 'promised', 'properly', 'quality', 'recommend', 'regret', 'reliable', 'resolved', 'respond', 'response', 'responsive', 'satisfied', 'seamless', 'service', 'slow', 'smooth', 'smoothly', 'start', 'still', 'stressfree', 'support', 'supposed', 'things', 'time', 'times', 'unacceptable', 'unhappy', 'unhelpful', 'unnecessary', 'unsatisfactory', 'use', 'using', 'valued', 'way', 'well', 'whole', 'without', 'worked', 'worst', 'would']]


In [78]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_numeric_scaled = scaler.fit_transform(X)

In [79]:
from sklearn.model_selection import train_test_split

# Target column: 'IsFraud'
y = df['sentiment_encoded'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_numeric_scaled,
    y,
    test_size=0.20,       # 20% test
    random_state=42,      # for reproducibility
    stratify=y            # optional but useful if class imbalance
)


In [81]:
from sklearn.ensemble import RandomForestClassifier

clf_rf = RandomForestClassifier(
    n_estimators=200,     # number of trees; adjust based on compute
    max_depth=None,       # let trees grow fully unless overfitting
    random_state=42,
    n_jobs=-1             # use all cores
)

clf_rf.fit(X_train, y_train)


RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [82]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# Predict on test set
y_pred = clf_lr.predict(X_test)      # or clf_rf
y_proba = clf_lr.predict_proba(X_test)[:, 1]  # probabilities for ROC AUC

# Basic metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1)
print("ROC AUC:", roc_auc)

# Detailed per-class report
print(classification_report(y_test, y_pred))


Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1: 1.0
ROC AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00       400

    accuracy                           1.00       800
   macro avg       1.00      1.00      1.00       800
weighted avg       1.00      1.00      1.00       800



In [84]:
# 1) feature names from the TF-IDF vectorizer
feature_names = tfidf.get_feature_names_out()

# 2) use the fitted logistic model you actually have
#    e.g., clf_lr instead of clf
coefs = clf_lr.coef_[0]

# 3) select only the TF-IDF part if you combined features
n_tfidf = len(feature_names)
tfidf_coefs = coefs[:n_tfidf]

# 4) rank and print top 10
import numpy as np
top_n = 10
top_indices = np.argsort(tfidf_coefs)[-top_n:][::-1]
top_features = feature_names[top_indices]
top_values = tfidf_coefs[top_indices]

print("Top TF-IDF words contributing to fraud prediction:")
for word, coef in zip(top_features, top_values):
    print(f"{word}: {coef:.4f}")


Top TF-IDF words contributing to fraud prediction:
choosing: 2.3054
happily: 0.7126
start: 0.5284
recommend: 0.4809
far: 0.4400
issues: 0.4400
still: 0.3038
felt: 0.3022
unhappy: 0.3022
handled: 0.3020
